# T0-bonus · Environment as code

## Goal

Replace notebook `00`'s portal/CLI environment creation with Terraform:
`powerplatform_environment` + `powerplatform_managed_environment` +
`powerplatform_environment_settings`, authenticated via OIDC so no secret
sits in a pipeline. Prove idempotency (`plan` shows zero diff on a
no-op re-run) and prove teardown (`destroy` actually removes it).


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
import subprocess
assert subprocess.run(["terraform", "version"], capture_output=True).returncode == 0, "install Terraform >= 1.8"
assert subprocess.run(["az", "version"], capture_output=True).returncode == 0, "install Azure CLI for OIDC login"
print("terraform + az CLI present")


This notebook assumes `infra/terraform/platform` — read `main.tf`, `variables.tf`, `dlp.tf`, `solution.tf` before running anything.


## Concept

Per finding #9, the `microsoft/power-platform` Terraform provider covers
`environment`, `managed_environment`, `environment_settings` (including
Copilot Studio settings), `environment_application_admin`, and `solution`
(roughly equivalent to `pac solution import`), with OIDC auth designed for
pipelines. This is the **platform layer** in the three-layer IaC model (see
repo README): it owns the environment, not the agent inside it and not the
Azure resources the agent's knowledge/tools reach — those are the other two
layers, `pac copilot` and Bicep respectively.

The idempotency requirement is not a nice-to-have here — it's the whole
point of choosing Terraform over a one-shot `pac admin create-environment`
script. A `plan` with no diff after a no-op run is the proof this is safe
to put in a pipeline that fires on every merge to `main`.


## Build


In [ ]:
import subprocess
tf_dir = "../infra/terraform/platform"

subprocess.run(["terraform", "init"], cwd=tf_dir, check=True)


In [ ]:
# az login --service-principal (or `az login` interactively for a local run)
# then plan, with the variables notebook 00 already decided on.
plan = subprocess.run(
    ["terraform", "plan",
     "-var", "environment_display_name=Contract Renewal Desk - dev",
     "-var", "location=unitedstates",
     "-var", "admin_app_object_id=$APP_CLIENT_OBJECT_ID"],
    cwd=tf_dir, capture_output=True, text=True,
)
print(plan.stdout[-3000:])


In [ ]:
apply = subprocess.run(["terraform", "apply", "-auto-approve",
                          "-var", "environment_display_name=Contract Renewal Desk - dev",
                          "-var", "location=unitedstates",
                          "-var", "admin_app_object_id=$APP_CLIENT_OBJECT_ID"],
                         cwd=tf_dir, capture_output=True, text=True)
print(apply.stdout[-3000:])


### Idempotency proof — plan again, expect zero diff


In [ ]:
plan2 = subprocess.run(["terraform", "plan", "-detailed-exitcode",
                          "-var", "environment_display_name=Contract Renewal Desk - dev",
                          "-var", "location=unitedstates",
                          "-var", "admin_app_object_id=$APP_CLIENT_OBJECT_ID"],
                         cwd=tf_dir)
assert plan2.returncode == 0, f"expected exit code 0 (no changes), got {plan2.returncode} — re-run created a diff"
print("idempotency proven: second plan is a no-op")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import subprocess
out = subprocess.run(["terraform", "output", "-json"], cwd=tf_dir, capture_output=True, text=True)
print(out.stdout)
# environment_id / environment_url should now match what csx.config expects
# in .env — update DATAVERSE_ENV_ID / DATAVERSE_ENV_URL from this output.


## Cost


In [ ]:
print("Environment creation itself does not meter Copilot Credits — only agent build/preview/test does (notebook 01 onward).")


## Teardown


In [ ]:
import subprocess
destroy = subprocess.run(["terraform", "destroy", "-auto-approve",
                            "-var", "environment_display_name=Contract Renewal Desk - dev",
                            "-var", "location=unitedstates",
                            "-var", "admin_app_object_id=$APP_CLIENT_OBJECT_ID"],
                           cwd=tf_dir, capture_output=True, text=True)
print(destroy.stdout[-2000:])
print("Only run this if you're tearing down the whole workshop environment — every later notebook depends on it existing.")
